# Seminar 5 - Federated Learning

#### Corentin LE BRIS (335776))  
#### Antoine METZ (335801)

This seminar reuses the Wi-Fi CSI pose classification problem, but with a federated learning setting. Instead of training one model on one central dataset, the training data are split between 10 clients. The server only aggregates model updates, while each client trains locally on its own data.

## 1. Imports

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report

# We keep the execution on CPU because the model and dataset are small enough for this seminar.
device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


## 2. ML model definition

We define a small 1D-CNN for the pose classification task. The input is a CSI vector, reshaped as 9 groups of 30 subcarriers. The convolution layers extract local patterns across the subcarriers, then the fully connected layers output one class among the 12 possible poses.

The model is intentionally simple. Since the goal is mainly to implement the federated learning pipeline, we do not need a very large architecture. Batch normalization helps stabilize training, and dropout is added to reduce overfitting during the local client updates.

In [2]:
class PoseCNN(nn.Module):
    def __init__(self, num_classes=12):
        super(PoseCNN, self).__init__()

        # Feature extraction on the CSI signal.
        self.conv_block = nn.Sequential(
            nn.Conv1d(in_channels=9, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        # Classification block.
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # The original vector is interpreted as 9 CSI groups and 30 subcarriers.
        x = x.view(-1, 9, 30)
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x

## 3. Federated learning setting

Here we define the main operations needed for federated learning. Each client loads its own local data, trains a copy of the current global model, and sends back only the updated weights. Then the server applies FedAvg to build the next global model.

At each communication round, we randomly select a subset of clients. This simulates the fact that in a real FL system, not every device is always available. The aggregation is weighted by the number of samples owned by each selected client, so larger local datasets have a larger influence on the global update.

In [3]:
directory = "client_datasets"

def load_client_data(client_id):
    feat_path = os.path.join(directory, f"client_{client_id}_features.csv")
    label_path = os.path.join(directory, f"client_{client_id}_labels.csv")

    X = pd.read_csv(feat_path, header=None).values.astype(np.float32)
    y = pd.read_csv(label_path, header=None).values.flatten().astype(np.int64)

    # Labels are converted from 1-12 to 0-11 for CrossEntropyLoss.
    y = y - 1
    return torch.tensor(X), torch.tensor(y)

def train_local_model(global_weights, X_local, y_local, epochs, batch_size, lr):
    model = PoseCNN().to(device)
    model.load_state_dict(global_weights)
    model.train()

    dataset = TensorDataset(X_local, y_local)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    return model.state_dict()

def federated_aggregation(local_models_weights, client_data_sizes):
    # FedAvg aggregation weighted by the local dataset size.
    total_data_points = sum(client_data_sizes)
    global_weights = {}

    first_client_weights = local_models_weights[0]

    for key in first_client_weights.keys():
        global_weights[key] = sum(
            local_models_weights[i][key] * (client_data_sizes[i] / total_data_points)
            for i in range(len(local_models_weights))
        )

    return global_weights

## 4. Federated training

We use 50 FL rounds with 2 local epochs per selected client. The local training is kept short because clients should not overfit too much on their own data before aggregation. At each round, 6 clients out of 10 are selected, which keeps the training collaborative without forcing all clients to participate every time.

In [4]:
# Hyperparameters
FL_ROUNDS = 50
LOCAL_EPOCHS = 2
BATCH_SIZE = 32
LEARNING_RATE = 0.005
NUM_CLIENTS = 10
CLIENTS_PER_ROUND = 6

global_model = PoseCNN().to(device)
global_weights = global_model.state_dict()

print("Beginning of training")

for round_idx in range(1, FL_ROUNDS + 1):
    local_weights_list = []
    client_sizes = []

    # Random subset of clients for the current round.
    selected_clients = np.random.choice(
        range(1, NUM_CLIENTS + 1),
        size=CLIENTS_PER_ROUND,
        replace=False
    ).tolist()

    for client_id in selected_clients:
        X_local, y_local = load_client_data(client_id)
        client_sizes.append(len(X_local))

        # Local update starting from the current global weights.
        local_weights = train_local_model(
            global_weights,
            X_local,
            y_local,
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE
        )
        local_weights_list.append(local_weights)

    # Server-side FedAvg update.
    global_weights = federated_aggregation(local_weights_list, client_sizes)
    print(f"Round {round_idx}/{FL_ROUNDS} done.")

# Load the final aggregated weights into the global model.
global_model.load_state_dict(global_weights)
print("Training completed")

Beginning of training
Round 1/50 done.
Round 2/50 done.
Round 3/50 done.
Round 4/50 done.
Round 5/50 done.
Round 6/50 done.
Round 7/50 done.
Round 8/50 done.
Round 9/50 done.
Round 10/50 done.
Round 11/50 done.
Round 12/50 done.
Round 13/50 done.
Round 14/50 done.
Round 15/50 done.
Round 16/50 done.
Round 17/50 done.
Round 18/50 done.
Round 19/50 done.
Round 20/50 done.
Round 21/50 done.
Round 22/50 done.
Round 23/50 done.
Round 24/50 done.
Round 25/50 done.
Round 26/50 done.
Round 27/50 done.
Round 28/50 done.
Round 29/50 done.
Round 30/50 done.
Round 31/50 done.
Round 32/50 done.
Round 33/50 done.
Round 34/50 done.
Round 35/50 done.
Round 36/50 done.
Round 37/50 done.
Round 38/50 done.
Round 39/50 done.
Round 40/50 done.
Round 41/50 done.
Round 42/50 done.
Round 43/50 done.
Round 44/50 done.
Round 45/50 done.
Round 46/50 done.
Round 47/50 done.
Round 48/50 done.
Round 49/50 done.
Round 50/50 done.
Training completed


## 5. Test evaluation

After the FL rounds, we evaluate the final global model on the common test partition. The test labels are only used at this stage to measure the performance.

In [5]:
def load_test_data():
    X_test = pd.read_csv("test_features.csv", header=None).values.astype(np.float32)
    y_test = pd.read_csv("test_labels.csv", header=None).values.flatten().astype(np.int64)
    y_test = y_test - 1
    return torch.tensor(X_test), torch.tensor(y_test)

X_test, y_test = load_test_data()

global_model.eval()

with torch.no_grad():
    X_test = X_test.to(device)
    outputs = global_model(X_test)
    _, predictions = torch.max(outputs, 1)

predictions = predictions.cpu().numpy()
y_test_np = y_test.numpy()

accuracy = accuracy_score(y_test_np, predictions)

print(f"Global Accuracy: {accuracy * 100:.2f}%
")

target_names = [f"Pose {i}" for i in range(1, 13)]
print(classification_report(y_test_np, predictions, target_names=target_names, zero_division=0))

Global Accuracy: 55.20%

              precision    recall  f1-score   support

      Pose 1       0.83      0.79      0.81        57
      Pose 2       0.60      0.18      0.27        51
      Pose 3       0.00      0.00      0.00        56
      Pose 4       0.50      0.53      0.52        47
      Pose 5       0.60      0.88      0.72        50
      Pose 6       0.58      0.72      0.64        47
      Pose 7       0.00      0.00      0.00        37
      Pose 8       0.35      0.76      0.48        17
      Pose 9       0.38      0.92      0.54        12
     Pose 10       0.69      0.75      0.72        63
     Pose 11       0.65      0.77      0.70        52
     Pose 12       0.15      0.73      0.25        11

    accuracy                           0.55       500
   macro avg       0.44      0.59      0.47       500
weighted avg       0.50      0.55      0.50       500



The global accuracy is about 55.2%. This is clearly lower than a simple centralized setting, but it is still meaningful because the model was trained through client updates instead of directly using one central training file. The classification report also shows that some poses are much easier to recognize than others. Poses 1, 5, 10 and 11 obtain reasonable F1-scores, while poses 3 and 7 are not correctly recovered in this run.

This suggests that the global model learns useful CSI patterns, but the federated setup is harder because each client only sees part of the data. More rounds, better client selection, or a different model could improve the result.

## Conclusion

In this notebook, we implemented the main steps of federated learning for Wi-Fi CSI pose classification. We defined a classification model, trained it locally on several clients, aggregated the local updates with FedAvg, and evaluated the final global model on the test set.

The experiment shows the basic idea of FL: the server does not need to centralize the raw client datasets. It only coordinates the training and combines the model updates. The final accuracy is not perfect, but the pipeline follows the expected FL process and gives a usable baseline for the seminar.